<a href="https://colab.research.google.com/github/DavidSenseman/BIO1173/blob/main/BIO1173_Class_06_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- BIO1173_CLASS_06_2:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **BIO 1173: Intro Computational Biology**

## **Module 6: Advanced Topics**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Module 6 Material

* Class_06_1: Reinforcement Learning
* **Class_06_2: ONNX Runtime Environment**
* Class_06_3: Analysis of DICOM images with Pytorch

## **Change your Runtime Now!**

For this lesson you must have a **GPU** hardware accelerator (e.g. T4 High-RAM).
NOTE: There is no need to use an "expensive" GPU like the A-100 for this lesson.

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of your work in case you need it later.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    from google.colab import auth
    auth.authenticate_user()
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your Google Drive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something similar to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.23.99.47",
  "hostname": "47.99.23.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your Keys

Run the next cell to test whether your `myUTSA_ID` and your `BIO1173_Key` are correctly installed in your Colab Secrets. You will need to have both keys correctly installed in your Colab Secrets in order to submit your work for grading using Electronic Submission.

In [ ]:
# @title Test Your Keys

from google.colab import userdata
import os

# Check if myUTSA_ID is properly loaded
try:
    # 1. Get the key from Secrets
    myUTSA_ID = userdata.get('myUTSA_ID')

    # 2. Set it as an environment variable
    os.environ['myUTSA_ID'] = myUTSA_ID

    # print("myUTSA_ID is loaded and environment variable set successfully!")
    print(f"myUTSA_ID: {myUTSA_ID}")

except Exception as e:
    print(f"Error loading myUTSA_ID: {e}")
    print("Please set your myUTSA_ID in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'myUTSA_ID'")
    print("3. Paste your myUTSA_ID and toggle 'Notebook access' on")

# Check if BIO1173 key is properly loaded
try:
    # 1. Get the key from Secrets
    BIO1173_KEY = userdata.get('BIO1173_KEY')

    # 2. Set it as an environment variable
    os.environ['BIO1173_KEY'] = BIO1173_KEY

    #print("BIO1173 key loaded and environment variable set successfully!")
    print(f"BIO1173_KEY: {BIO1173_KEY}")

except Exception as e:
    print(f"Error loading BIO1173 key: {e}")
    print("Please set your BIO1173 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'BIO1173_KEY'")
    print("3. Paste your BIO1173 key and toggle 'Notebook access' on")

If you have correctly installed your myUTSA id in your Colab Secrets, you should see something similar to following but with your information:

myUTSA_ID: abc123
BIO1173_KEY: BIO1173-F26-04-999-ABC-DEF

However, if you see an error message, you will need to fix this problem before you can submit your lesson for grading. Ask your Instructor or TA for help if you can't resolve the problem yourself --- that's what they are here for, to help you with course problems.

## Accelerated Run-time Check

Run the following code cell to check what hardware acceleration you are using. To run this lesson, you must be running a Graphics Processing Unit (GPU) such as the `T4` with high ram enabled.

In [ ]:
# @title Accelerated Run-time Check

import torch

# Check for GPU
def check_colab_gpu():
    print("=== Colab GPU Check ===")

    # Check PyTorch
    pt_gpu = torch.cuda.is_available()
    print(f"PyTorch GPU available: {pt_gpu}")

    if pt_gpu:
        print(f"PyTorch device count: {torch.cuda.device_count()}")
        print(f"PyTorch current device: {torch.cuda.current_device()}")
        print(f"PyTorch device name: {torch.cuda.get_device_name()}")
        print("You are good to go!")

    else:
        print("No compatible device found")
        print("WARNING: You must run this assigment using either a GPU to earn credit")
        print("Change your RUNTIME now and start over!")

check_colab_gpu()

If your Runtime is correct, you should see something _similar_ to the following output:

```text
=== Colab GPU Check ===
PyTorch GPU available: True
PyTorch device count: 1
PyTorch current device: 0
PyTorch device name: Tesla T4
You are good to go!
```

# **ONNX Runtime Environment**

### ONNX Runtime Overview

**ONNX Runtime** is a high-performance inference engine developed by Microsoft for executing models in the **Open Neural Network Exchange (ONNX)** format. It is designed to be **cross-platform**, **language-agnostic**, and **hardware-optimized**, supporting execution on CPUs, GPUs, and specialized accelerators like `NVIDIA` **TensorRT** and `Intel` **OpenVINO**.

ONNX Runtime is particularly useful in scenarios where:
- **Interoperability** is needed across different frameworks (e.g., PyTorch, TensorFlow, scikit-learn).
- **Deployment efficiency** is critical, offering faster inference times and reduced resource consumption.
- **Portability** is a priority, allowing models to be deployed in cloud, edge, and mobile environments.
- **Hardware acceleration** is desired, with built-in support for various execution providers.

By decoupling model training from inference, ONNX Runtime enables developers to train models in their preferred framework and deploy them in a streamlined, optimized runtime environment.



## Install ONNX library packages

Run the code in the cell below to install the `ONNX Runtime` package.

In [ ]:
# @title Install ONNX library packages

!pip install -q onnxruntime
!pip install -q onnxscript

print("ONNX library packages have been installed. ✅")

If the code is correct you should see something _similar_ to the following output:
```text
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 17.1 MB/s eta 0:00:00
ONNX library packages have been installed. ✅
```

## Example 1: Copy PyTorch Model from Google Drive

In `Class_03_2` you trained a PyTorch model callled `ResNet50_model_244` on the `Diabetic Retinopathy` image dataset and saved it to your Google Drive.

The code in the cell below copies your saved Pytorch model to your current Colab directory.

**NOTE:** Contact your Instructor for help if you don't have your saved neural network.

In [ ]:
# @title Example 1: Copy PyTorch Model from Google Drive

import os
import shutil

# ------------------------------------------------------------------
#  1. Define the model filename (only the name, not the full path)
# ------------------------------------------------------------------
model_folder_eg = 'ResNet50_model_244'
model_filename_eg = 'ResNet50_model_244.pth'

# ------------------------------------------------------------------
#  2. Build absolute paths
# ------------------------------------------------------------------
gdrive_model_path = os.path.join('/content/drive/MyDrive', model_folder_eg, model_filename_eg)
local_model_path_eg  = os.path.join('/content', model_filename_eg)

# ------------------------------------------------------------------
#  3. Check that the source file exists
# ------------------------------------------------------------------
if not os.path.exists(gdrive_model_path):
    print(f"[ERROR ❌] Source file not found:\n  {gdrive_model_path}\n"
          "Please make sure the file exists in the specified Google Drive folder "
          "and that the Drive is mounted.")
else:
    # ------------------------------------------------------------------
    #  4. Attempt to copy the file with error handling
    # ------------------------------------------------------------------
    try:
        shutil.copy(gdrive_model_path, local_model_path_eg)
        print(f"[SUCCESS 🏆] PyTorch model '{model_filename_eg}' was copied from "
              f"Google Drive to your current Colab directory:\n  {local_model_path_eg}")
    except FileNotFoundError as fnf_err:
        # This is a rare case – we already checked existence, but handle it anyway
        print(f"[ERROR ❌] FileNotFoundError during copy: {fnf_err}")
    except PermissionError as perm_err:
        print(f"[ERROR ❌] Permission denied while copying:\n  {perm_err}")
    except OSError as os_err:
        # Covers other OS related errors (e.g., disk full, invalid characters)
        print(f"[ERROR ❌] OS error during copy: {os_err}")
    except Exception as exc:
        # Fallback for any other unexpected exception
        print(f"[ERROR ❌] Unexpected error: {exc}")

If your ResNet50_model_244 is on your Google Drive, you should see the following output:
```text
[SUCCESS 🏆] PyTorch model 'ResNet50_model_244.pth' was copied from Google Drive to your current Colab directory:
  /content/ResNet50_model_244.pth
```

If you didn't save your model or you deleted it, contact your Instructor for help. You will not be able to complete this class lesson without your saved model.

## Example 2: Load PyTorch Model

The code in the cell below loads the PyTorch model and determines its input shape. The suffix `_eg` is added to the name of the loaded model (`model_eg`) as well as other model attributes. As before, this has been done to keep these variables separate from similar variables that you will generate later in the **Exercises**.


In [ ]:
# @title Example 2: Load PyTorch model

import os
import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------------
# 1. Load the state dict and inspect the custom head
# ------------------------------------------------------------------
print("Inspecting saved weights...")
state_dict = torch.load(local_model_path_eg, map_location=torch.device('cpu'))

# Read the shape of the custom fc layers from the saved weights
fc0_in  = state_dict['fc.0.weight'].shape[1]   # input features
fc0_out = state_dict['fc.0.weight'].shape[0]   # hidden layer size
fc3_out = state_dict['fc.3.weight'].shape[0]   # number of output classes

print(f"  fc.0 : Linear({fc0_in}, {fc0_out})")
print(f"  fc.1 : ReLU")
print(f"  fc.2 : Dropout")
print(f"  fc.3 : Linear({fc0_out}, {fc3_out})")
print(f"  → Detected {fc3_out} output classes")

# ------------------------------------------------------------------
# 2. Build ResNet50 and replace fc with the matching custom head
# ------------------------------------------------------------------
print("\nLoading PyTorch model...")
model_eg = models.resnet50()

model_eg.fc = nn.Sequential(
    nn.Linear(fc0_in, fc0_out),
    nn.ReLU(),
    nn.Dropout(),
    nn.Linear(fc0_out, fc3_out)
)

# ------------------------------------------------------------------
# 3. Load weights — should now match perfectly
# ------------------------------------------------------------------
model_eg.load_state_dict(state_dict)
model_eg.eval()
print(f"Model {model_filename_eg} loaded successfully!")

# ------------------------------------------------------------------
# 4. Test forward pass
# ------------------------------------------------------------------
test_input = torch.randn(1, 3, 244, 244)
with torch.no_grad():
    output = model_eg(test_input)
print(f"Test input shape : {test_input.shape}")
print(f"Output shape     : {output.shape}")

If the code is correct you should see the following output

```text
Inspecting saved weights...
  fc.0 : Linear(2048, 256)
  fc.1 : ReLU
  fc.2 : Dropout
  fc.3 : Linear(256, 5)
  → Detected 5 output classes

Loading PyTorch model...
Model ResNet50_model_244.pth loaded successfully!
Test input shape : torch.Size([1, 3, 244, 244])
Output shape     : torch.Size([1, 5])
```

## Example 3: Convert PyTorch Model to ONNX Format

The code in the cell below converts the PyTorch model into the ONNX format.

In [ ]:
# Example 3: Convert PyTorch model to ONNX format

import torch
import warnings
import logging

# Suppress the three warning sources
warnings.filterwarnings("ignore")                         # Suppresses UserWarning and FutureWarning
logging.getLogger("onnxscript").setLevel(logging.ERROR)   # Suppresses onnxscript WARNING logs
torch._logging.set_logs(onnx=logging.ERROR)               # Suppresses [torch.onnx] info logs

print("Converting PyTorch model to ONNX format...")
try:
    input_shape_eg = (1, 3, 244, 244)
    dummy_input = torch.randn(input_shape_eg)

    onnx_filename_eg = model_filename_eg.replace('.pth', '.onnx')
    output_path = f"/content/{onnx_filename_eg}"

    torch.onnx.export(
        model_eg,
        dummy_input,
        output_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input":  {0: "batch_size"},
            "output": {0: "batch_size"}
        },
        opset_version=18          # ← matches what PyTorch actually uses, eliminates version warnings
    )

    print(f"Model {model_filename_eg} converted successfully!")
    print(f"ONNX model saved to: {output_path}")

except Exception as e:
    print(f"Conversion failed with error: {e}")

finally:
    # Restore normal warning behaviour for the rest of the notebook
    warnings.filterwarnings("default")
    logging.getLogger("onnxscript").setLevel(logging.WARNING)
    torch._logging.set_logs(onnx=logging.WARNING)

If the code is correct, you should see something _similar_ to the following output:
```text
Converting PyTorch model to ONNX format...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 107 of general pattern rewrite rules.
Model ResNet50_model_244.pth converted successfully!
ONNX model saved to: /content/ResNet50_model_244.onnx
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
```

## Example 4: Verify ONNX Model

The code in the cell below performs **validation and inspection** of the ONNX model file. Checking is accomplished by the following line of code:
```text
onnx.checker.check_model(onnx_model_loaded_eg)
```
If no errors are raised, the model is considered valid.

In [ ]:
# @title Example 4: Verify ONNX model

import onnx

# Load and check the ONNX model
onnx_filename_eg = model_filename_eg.replace('.pth', '.onnx')
output_path = f"/content/{onnx_filename_eg}"

# Load and check the ONNX model
onnx_model_loaded_eg = onnx.load(output_path)
onnx.checker.check_model(onnx_model_loaded_eg)
print("ONNX model validation successful!")

# Display ONNX model information
print("\nONNX Model Information:")
print(f"Model name: {onnx_model_loaded_eg.graph.name}")
print(f"Number of inputs: {len(onnx_model_loaded_eg.graph.input)}")
print(f"Number of outputs: {len(onnx_model_loaded_eg.graph.output)}")

If the code is correct you should see the following output
```text
ONNX model validation successful!

ONNX Model Information:
Model name: main_graph
Number of inputs: 1
Number of outputs: 1
```
This confirms that your ONNX model was loaded without errors and passes all ONNX format validations.


## Example 5: Model Comparison

The code in the cell below performs a comparison between predictions from the PyTorch model and its ONNX-converted counterpart to verify that the conversion preserved the model's behavior.

This code runs inference with both models
```text
onnx_pred_eg = onnx_session_eg.run([output_name_eg], {input_name_eg: test_input_eg})
pytorch_pred_eg = model_eg.predict(test_input_eg, verbose=0)
```
And then the predictions are compared.



In [ ]:
# @title Example 5: Model comparison

import onnxruntime as ort
import numpy as np
import torch

# Load the ONNX model
onnx_filename_eg = model_filename_eg.replace('.pth', '.onnx')
output_path = f"/content/{onnx_filename_eg}"
onnx_session_eg = ort.InferenceSession(output_path)

# Get input/output names
input_name_eg = onnx_session_eg.get_inputs()[0].name
output_name_eg = onnx_session_eg.get_outputs()[0].name

# Create test data - PyTorch uses (batch, channels, height, width) ordering
test_input_np_eg = np.random.randn(1, 3, 244, 244).astype(np.float32)

# Run prediction with ONNX model
onnx_pred_eg = onnx_session_eg.run([output_name_eg],
               {input_name_eg: test_input_np_eg})

# Run prediction with PyTorch model
test_input_tensor_eg = torch.tensor(test_input_np_eg)
with torch.no_grad():
    pytorch_pred_eg = model_eg(test_input_tensor_eg).numpy()

# Compare results
print("ONNX prediction shape:", onnx_pred_eg[0].shape)
print("PyTorch prediction shape:", pytorch_pred_eg.shape)
print("Max difference:", np.max(np.abs(onnx_pred_eg[0] - pytorch_pred_eg)))
print("Mean difference:", np.mean(np.abs(onnx_pred_eg[0] - pytorch_pred_eg)))

# Verify they're essentially identical
are_identical = np.allclose(onnx_pred_eg[0], pytorch_pred_eg, atol=1e-5)
print("Results are identical (within floating point precision):", are_identical)

if not are_identical:
    print("Note: Small numerical differences may occur due to floating-point precision.")

If the code is correct you should see something _similar_ to the following output

```text
ONNX prediction shape: (1, 5)
PyTorch prediction shape: (1, 5)
Max difference: 1.1920929e-06
Mean difference: 5.722046e-07
Results are identical (within floating point precision): True
```

Both models are virtually identical!

## Example 6: Visualize Predicted Outputs

The code in the cell below generates a bar chart showing the predicted outputs from the `PyTorch` and the `ONNX` models.

In [ ]:
# @title Example 6: Visualize predicted outputs

import numpy as np
import matplotlib.pyplot as plt
import torch

# Run predictions using the same test input from Example 5
with torch.no_grad():
    pytorch_pred_eg = model_eg(torch.tensor(test_input_np_eg)).numpy().flatten()

onnx_pred_eg = onnx_session_eg.run([output_name_eg], {input_name_eg: test_input_np_eg})[0].flatten()

# Create class labels
num_classes_eg = len(pytorch_pred_eg)
labels = [f"Class {i}" for i in range(num_classes_eg)]

# Plotting
x = np.arange(num_classes_eg)
width = 0.35

plt.figure(figsize=(7, 5))
plt.bar(x - width/2, pytorch_pred_eg, width, label='PyTorch')
plt.bar(x + width/2, onnx_pred_eg, width, label='ONNX')
plt.xlabel('Classes')
plt.ylabel('Raw Prediction Score')
plt.title('Comparison of PyTorch and ONNX Model Predictions')
plt.xticks(x, labels)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

If the code is correct you should see the following output

![__](https://biologicslab.co/BIO1173/images/class_06/class_06_2_image01Z.png)

The graph shows how closely the ONNX model replicates the behavior of the original PyTorch model across five classes.

## Example 7: Generate Random Testing Data

The code in the cell below generates random data that will be used in the next step to compare the accuracy of the converted ONNX model against the accuracy of the original PyTorch model.

Note that value of the `seed` is specified by a variable called `seed_val` that is defined by the user.

In [ ]:
# @title Example 7: Generate random testing data

# Generate random test data for model comparison
import numpy as np
import torch
import torch.nn.functional as F

# Set value for random seed
seed_val = 42
print(f"Random seed set to {seed_val}")

# Set random seed for reproducibility
np.random.seed(seed_val)
torch.manual_seed(seed_val)

# Your specific model parameters
input_shape_eg = (1, 3, 244, 244)  # PyTorch uses (batch, channels, height, width)
num_classes_eg = 5                  # Based on your output shape (1, 5)

# Generate random test data
print(f"Generating test data with input shape: {input_shape_eg}")
print(f"Number of samples: 100")

# Create random input data (matching your model's expected input)
X_test_eg = np.random.randn(100, *input_shape_eg[1:]).astype(np.float32)

# Generate random labels for 5-class classification
y_test_eg = np.random.randint(0, num_classes_eg, (100,))

# One-hot encode using PyTorch instead of tf.keras.utils.to_categorical
y_test_onehot_eg = F.one_hot(torch.tensor(y_test_eg), num_classes=num_classes_eg).numpy()

print(f"Generated X_test shape: {X_test_eg.shape}")
print(f"Generated y_test shape: {y_test_onehot_eg.shape}")

# Print sample data to verify
print(f"Sample input data (first 5 samples): {X_test_eg[:5].flatten()[:10]}")
print(f"Sample labels (first 5): {y_test_eg[:5]}")

print("Test data generation complete!")

If the code is correct you should see the following output

```text
Random seed set to 42
Generating test data with input shape: (1, 3, 244, 244)
Number of samples: 100
Generated X_test shape: (100, 3, 244, 244)
Generated y_test shape: (100, 5)
Sample input data (first 5 samples): [ 0.49671414 -0.1382643   0.64768857  1.5230298  -0.23415338 -0.23413695
  1.5792128   0.7674347  -0.46947438  0.54256004]
Sample labels (first 5): [2 2 2 2 2]
Test data generation complete!
```

## Example 8: Compute Relative Accuracy

This code in the cell below performs a two-part evaluation of a `PyTorch model` and its `ONNX-converted version`:

#### **Part 1: Single-Sample Equivalence Check**
* **Purpose:** To verify that both models produce nearly identical predictions for a single input.
* **Steps:**
1. Load the `ONNX model` and prepare an inference session.
2. Extract input/output tensor names.
3. Select one sample from the test dataset (X_test[0:1]).
4. Run predictions using both models.
5. Compare the outputs using:
* Shape
* Max and mean absolute differences
* `np.allclose()` to check numerical equivalence within a tolerance.

#### **Part 2: Accuracy Comparison on Full Test Set**
* **Purpose:** To compare the classification accuracy of both models on the entire test dataset.
* **Steps:**
1. Run predictions on X_test using both models.
2. Convert predicted probabilities to class labels using np.argmax().
3. Compare predicted labels to true labels (y_test_onehot).
4. Compute accuracy for each model using np.mean(predicted == true).
5. Print the accuracy values and their difference.
6. Display sample predictions for manual inspection.

##### **Why This Is Useful**
* Ensures the `ONNX` model is a faithful representation of the original `PyTorch` model.
* Helps detect any discrepancies introduced during model conversion.
* Validates that the ONNX model is suitable for deployment without loss of performance.


In [ ]:
# @title Example 8: Compute relative accuracy

import torch
import numpy as np

# Load the ONNX model
onnx_filename_eg = model_filename_eg.replace('.pth', '.onnx')
output_path_eg = f"/content/{onnx_filename_eg}"
onnx_session_eg = ort.InferenceSession(output_path_eg)

# Get input/output names
input_name_eg = onnx_session_eg.get_inputs()[0].name
output_name_eg = onnx_session_eg.get_outputs()[0].name

# First: Verify the models produce identical results on a single sample
print("Verifying model equivalence on single sample...")
test_sample_eg = X_test_eg[0:1]  # Take first sample to maintain batch dimension

onnx_pred_eg = onnx_session_eg.run([output_name_eg], {input_name_eg: test_sample_eg})
with torch.no_grad():
    pytorch_pred_eg = model_eg(torch.tensor(test_sample_eg)).numpy()

print("ONNX prediction shape:", onnx_pred_eg[0].shape)
print("PyTorch prediction shape:", pytorch_pred_eg.shape)
print("Max difference:", np.max(np.abs(onnx_pred_eg[0] - pytorch_pred_eg)))
print("Mean difference:", np.mean(np.abs(onnx_pred_eg[0] - pytorch_pred_eg)))

are_identical = np.allclose(onnx_pred_eg[0], pytorch_pred_eg, atol=1e-5)
print("Results are identical (within floating point precision):", are_identical)

# Second: Compare accuracies using your generated test data
try:
    print("\nComparing model accuracies on the generated test dataset...")

    # Get predictions from PyTorch model
    print("Getting predictions from PyTorch model...")
    with torch.no_grad():
        pytorch_pred2_eg = model_eg(torch.tensor(X_test_eg)).numpy()

    # Get predictions from ONNX model
    print("Getting predictions from ONNX model...")
    onnx_pred2_eg = onnx_session_eg.run([output_name_eg], {input_name_eg: X_test_eg})

    # Convert logits to class labels for accuracy calculation
    pytorch_pred_classes_eg = np.argmax(pytorch_pred2_eg, axis=1)
    onnex_pred_classes_eg = np.argmax(onnx_pred2_eg[0], axis=1)

    # Use your generated labels
    y_test_classes_eg = np.argmax(y_test_onehot_eg, axis=1)

    # Calculate accuracies
    pytorch_accuracy_eg = np.mean(pytorch_pred_classes_eg == y_test_classes_eg)
    onnx_accuracy_eg = np.mean(onnex_pred_classes_eg == y_test_classes_eg)

    print(f"PyTorch model accuracy: {pytorch_accuracy_eg:.4f}")
    print(f"ONNX model accuracy: {onnx_accuracy_eg:.4f}")
    print(f"Difference in accuracy: {abs(pytorch_accuracy_eg - onnx_accuracy_eg):.6f}")

    # Additional verification
    print("\n--- Verification ---")
    print(f"Number of test samples: {len(y_test_classes_eg)}")
    print(f"PyTorch predictions shape: {pytorch_pred2_eg.shape}")
    print(f"ONNX predictions shape: {onnx_pred2_eg[0].shape}")

    # Show some sample comparisons
    print("\nSample predictions comparison:")
    for i in range(5):
        print(f"Sample {i}: True={y_test_classes_eg[i]}, PyTorch={pytorch_pred_classes_eg[i]}, ONNX={onnex_pred_classes_eg[i]}")

except Exception as e:
    print(f"Error during accuracy comparison: {e}")
    print("Make sure you have defined X_test_eg, y_test_onehot_eg, and model_eg before running this code")
    if 'X_test_eg' in locals():
        print(f"X_test shape: {X_test_eg.shape}")
    if 'y_test_onehot_eg' in locals():
        print(f"y_test_onehot shape: {y_test_onehot_eg.shape}")

If the code is correct you should see the following output

```text
Verifying model equivalence on single sample...
ONNX prediction shape: (1, 5)
PyTorch prediction shape: (1, 5)
Max difference: 1.0430813e-07
Mean difference: 5.5134297e-08
Results are identical (within floating point precision): True

Comparing model accuracies on the generated test dataset...
Getting predictions from PyTorch model...
Getting predictions from ONNX model...
PyTorch model accuracy: 0.2300
ONNX model accuracy: 0.2300
Difference in accuracy: 0.000000

--- Verification ---
Number of test samples: 100
PyTorch predictions shape: (100, 5)
ONNX predictions shape: (100, 5)

Sample predictions comparison:
Sample 0: True=2, PyTorch=2, ONNX=2
Sample 1: True=2, PyTorch=2, ONNX=2
Sample 2: True=2, PyTorch=2, ONNX=2
Sample 3: True=2, PyTorch=2, ONNX=2
Sample 4: True=2, PyTorch=2, ONNX=2
```

## Example 9: Visualize Similarities with Confusion Plots

The code in the cell below generates a Confusion Plot for the PyTorch model (left) and the ONNX model (right).

In [ ]:
# @title Example 9: Visualize Similarities with Confusion Plots

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix

# Get predictions
with torch.no_grad():
    pytorch_pred_eg = model_eg(torch.tensor(X_test_eg)).numpy()
onnx_pred_eg = onnx_session_eg.run([output_name_eg], {input_name_eg: X_test_eg})

pytorch_pred_classes_eg = np.argmax(pytorch_pred_eg, axis=1)
onnex_pred_classes_eg = np.argmax(onnx_pred_eg[0], axis=1)

# Handle y_test formatting - check if it's one-hot encoded or already class labels
if len(y_test_eg.shape) > 1 and y_test_eg.shape[1] > 1:
    y_test_classes_eg = np.argmax(y_test_eg, axis=1)
else:
    y_test_classes_eg = y_test_eg

# Compute confusion matrices
cm_pytorch = confusion_matrix(y_test_classes_eg, pytorch_pred_classes_eg)
cm_onnx = confusion_matrix(y_test_classes_eg, onnex_pred_classes_eg)

# Set plot style
plt.style.use('seaborn-v0_8')

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(5, 5))

# Plot PyTorch confusion matrix
sns.heatmap(cm_pytorch, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('PyTorch Model Confusion Matrix')
axes[0].set_xlabel('Predicted Labels')
axes[0].set_ylabel('True Labels')

# Plot ONNX confusion matrix
sns.heatmap(cm_onnx, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('ONNX Model Confusion Matrix')
axes[1].set_xlabel('Predicted Labels')
axes[1].set_ylabel('True Labels')

# Adjust layout
plt.tight_layout()
plt.show()

If the code is correct you should see something _similar_ to the following output

![__](https://biologicslab.co/BIO1173/images/class_06/class_06_2_image02Z.png)

### **Interpretation**

The confusion plot shows that your ONNX‐converted model reproduces the original PyTorch model’s predictions almost exactly. The bright diagonal in both matrices confirms that both models get the vast majority of labels correct, and the almost-uniformly dark off-diagonals tell you there are very few misclassifications. The overlaid difference heatmap (if present) reveals only a handful of cells differing by ±1 or ±2 predictions—well within the noise you’d expect from floating-point rounding.

#### **What the Diagonals Tell You**

* Each cell on the diagonal represents “true class X predicted as X.”

* Both PyTorch and ONNX show high counts here, indicating strong per-class accuracy.

* No entire row or column dramatically degrades when you switch runtimes, so no class has suddenly become “invisible” to inference.

#### **What the Off-Diagonals Reveal**

* Off-diagonal entries are where misclassifications live.

* Sparse, low-value cells in both matrices confirm that the error patterns are essentially identical.

* Any small red/blue tint in the difference plot signals ±1–2 sample swaps between classes (e.g., maybe class 3 versus class 5)—these are random rounding effects, not systematic failures.

#### **Key Takeaways**

1. **Conversion Fidelity:**
The ONNX export preserved the decision boundaries. There’s no systemic bias introduced by the conversion.

2. **Numerical Drift:**
Tiny count shifts stem from backend differences in floating-point implementations (Softmax, logits ordering, etc.). They’re negligible for production.

3. **Action Items**

* If you need bit-perfect parity, consider aligning the PyTorch and ONNX runtimes’ epsilon or Softmax parameters.

* Run a full classification report (precision, recall, F1) on both to quantify any micro-differences.

## Example 10: Save ONNX Model to Google Drive

Run the next cell to save your retrained `ResNet50_model_244.onnx` model to your Google Drive.

**IMPORTANT NOTE** You will be using this saved ONNX model in a latter class lesson so make sure **not** to delete it from your Google Drive!

In [ ]:
# @title Example 10: Save ONNX Model to Google Drive

import os
import shutil

# --------------------------------------------------------------
# 1️⃣  Mount Google Drive (do this only once per session)
# --------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')

# --------------------------------------------------------------
# 2️⃣  Define the names / paths
# --------------------------------------------------------------
model_name     = "ResNet50_model_244"                     # model object name (without extension)
gdrive_dir     = f"/content/drive/My Drive/{model_name}"  # folder on Drive
gdrive_file    = f"{gdrive_dir}.onnx"                     # the ONNX file we want to keep

local_file     = f"/content/{model_name}.onnx"            # local ONNX file to copy

# --------------------------------------------------------------
# 3️⃣  Make sure the Drive folder exists
# --------------------------------------------------------------
os.makedirs(gdrive_dir, exist_ok=True)

# --------------------------------------------------------------
# 4️⃣  Copy the existing ONNX model from local storage to Drive
# --------------------------------------------------------------
if os.path.exists(local_file):
    print(f"Copying {local_file} to Google Drive...")
    shutil.copy2(local_file, gdrive_file)
    print("ONNX model copied successfully!")
else:
    print(f"Error: {local_file} not found!")

# --------------------------------------------------------------
# 5️⃣  OPTIONAL: Verify the Drive copy exists
# --------------------------------------------------------------
print("Drive copy present:", os.path.exists(gdrive_file))

# List files in Drive to verify
!ls -lh "/content/drive/MyDrive"

If the code is correct, you should see something _similar_ to the following output:
```
Copying /content/ResNet50_model_244.onnx to Google Drive...
ONNX model copied successfully!
Drive copy present: True
total 2.5G
drwx------ 2 root root 4.0K Apr  5  2024 'Colab Notebooks'
drwx------ 2 root root 4.0K Aug 18  2025  ResNet101_model_512
drwx------ 2 root root 4.0K Aug 21  2025  ResNet50_model_244
drwx------ 2 root root 4.0K Apr 12 18:30  ResNet50_model_244.onnx

```

---

# **Exercises**

For the Exercises, you are to repeat the Examples but instead of converting your saved PyTorch model `ResNet50_model_244` to ONNX, you are to convert your saved PyTorch model `ResNet101_model_512` to ONNX.

As above, you didn't save your `ResNet101_model_512` to your Google Drive, or you have deleted it, contact your Instructor for help.

## **Exercise 1: Download Saved PyTorch File**

In the cell below write the code to copy the PyTorch model `ResNet101_model_512.pth` from your Google Drive to you current Colab directory. If you don't have your `ResNet101_model_512` saved to your Google Drive, you need to contact your Instructor or TA for help.

**Code Hints:**

1. Copy-and-paste Example 1 into the cell below.
2. Change the model filename using this code chunk:
```Python
# ------------------------------------------------------------------
#  1. Define the model filename (only the name, not the full path)
# ------------------------------------------------------------------
model_folder_ex = 'ResNet101_model_512'
model_filename_ex = 'ResNet101_model_512.pth'
```
3. Change every instance of the suffix `_eg` to `_ex`.

In [ ]:
# @title Exercise 1: Copy PyTorch Model from Google Drive




If your code is correct, you should see the following output:
```text
[SUCCESS 🏆] PyTorch model 'ResNet101_model_512.pth' was copied from Google Drive to your current Colab directory:
  /content/ResNet101_model_512.pth
```

If you get an error message, it probably means that you don't have the file `ResNet101_model_512.pth` on your Google drive. Please contact your Instructor or TA for assistance.


## **Exercise 2: Load PyTorch Model**

In the cell below write the code to load the PyTorch model and determines its input shape.

**Code Hints:**

1. Copy-and-paste Example 2 into the cell below.

2. Change the model name using this code chunk:
```Python
# ------------------------------------------------------------------
# 2. Build ResNet101 and replace fc with the matching custom head
# ------------------------------------------------------------------
print("\nLoading PyTorch model...")
model_ex = models.resnet101()
```

3. Change the input shape to read as follows:
```Python
test_input = torch.randn(1, 3, 512, 512)
```
4. Change every instance of the suffix `_eg` to `_ex`.


In [ ]:
# @title Exercise 2: Load PyTorch model



If the code is correct you should see the following output

```text

model_ex.load_state_dict(state_dict)
model_ex.eval()
print(f"Model {model_filename_ex} loaded successfully!")

# ------------------------------------------------------------------
# 4. Test forward pass
# ------------------------------------------------------------------
test_input = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    output = model_ex(test_input)

Inspecting saved weights...
  fc.0 : Linear(2048, 256)
  fc.1 : ReLU
  fc.2 : Dropout
  fc.3 : Linear(256, 5)
  → Detected 5 output classes

Loading PyTorch model...
Model ResNet101_model_512.pth loaded successfully!
Test input shape : torch.Size([1, 3, 512, 512])
Output shape     : torch.Size([1, 5])
```


## **Exercise 3: Convert PyTorch Model to ONNX Format**

In the cell below write the code to convert the PyTorch model into the ONNX format. For credit you need to make sure that all the variables with the `_eg` suffix have been renamed to use the `_ex` suffix.

**Code Hints:**
1. Copy-and-paste Example 3 into the cell below.
2. Change the variable `input_shape_ex` to read as follows:

```Python
print("Converting PyTorch model to ONNX format...")
try:
    input_shape_ex = (1, 3, 512, 512)
    dummy_input = torch.randn(input_shape_ex)
    input_shape_ex = (1, 3, 512, 512)
    dummy_input = torch.randn(input_shape_ex)
```
3. Change every instance of the suffix `_eg` to `_ex`.

In [ ]:
# @title Exercise 3: Convert PyTorch model to ONNX format



If the code is correct you should see the following output

```text
Converting PyTorch model to ONNX format...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 209 of general pattern rewrite rules.
Model ResNet101_model_512.pth converted successfully!
ONNX model saved to: /content/ResNet101_model_512.onnx
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
```

## **Exercise 4: Verify ONNX Model**

In the cell below write the code to perform validation and inspection of the ONNX model file.

**Code Hints:**
1. Copy-and-paste Example 4 into the cell below.
2. Change every instance of the suffix `_eg` with `_ex`.

In [ ]:
# @title Exercise 4: Verify ONNX model




If the code is correct you should see the following output

```text
ONNX model validation successful!

ONNX Model Information:
Model name: main_graph
Number of inputs: 1
Number of outputs: 1
```


## **Exercise 5: Model Comparison**

In the cell below write the code to perform a comparison between predictions from the Python model and its ONNX-converted counterpart to verify that the conversion preserved the model's behavior.


**Code Hints:**
1. Copy-and-paste Example 5 into the cell below.
2. Change the test data to match your larger model using this code chunk:
```Python
# Create test data - PyTorch uses (batch, channels, height, width) ordering
test_input_np_ex = np.random.randn(1, 3, 512, 512).astype(np.float32)
```
3. Change every instance of the suffix `_eg` to `_ex`.




In [ ]:
# @title Exercise 5: Model comparison




If the code is correct you should see something _similar_ to the following output

```text
ONNX prediction shape: (1, 5)
PyTorch prediction shape: (1, 5)
Max difference: 2.1457672e-06
Mean difference: 1.3887882e-06
Results are identical (within floating point precision): True
```

## **Exercise 6: Visualize Predicted Outputs**

In the cell below write the code to generate a bar chart showing the predicted outputs from the `PyTorch` and the `ONNX` models.

In [ ]:
# @title Example 6: Visualize predicted outputs




If the code is correct you should see something _similar_ to the following output:

![__](https://biologicslab.co/BIO1173/images/class_06/class_06_2_image03Z.png)

The graph shows how closely the ONNX model replicates the behavior of the original PyTorch model across five classes. The ONNX model closely mirrors the PyTorch model's output, which is a good sign that the conversion preserved the model's behavior.

## **Exercise 7: Generate Random Testing Data**

In the cell below write the code to generate random data that will be used in the next step to compare the accuracy of the converted ONNX model against the accuracy of the original PyTorch model.

Note that value of the `seed` is specified by a variable called `seed_val` that is defined by the user. Make sure you leave the random seed set to `42`.

**Code Hints:**

1. Change the model parameters as shown in this code chunk:
```Python
# Your specific model parameters
input_shape_ex = (1, 3, 512, 512)  # PyTorch uses (batch, channels, height, width)
num_classes_ex = 5                  # Based on your output shape (1, 5)

```

In [ ]:
# @title Exercise 7: Generate random testing data




If the code is correct you should see the following output

```text
Random seed set to 42
Generating test data with input shape: (1, 3, 512, 512)
Number of samples: 100
Generated X_test shape: (100, 3, 512, 512)
Generated y_test shape: (100, 5)
Sample input data (first 5 samples): [ 0.49671414 -0.1382643   0.64768857  1.5230298  -0.23415338 -0.23413695
  1.5792128   0.7674347  -0.46947438  0.54256004]
Sample labels (first 5): [0 1 0 3 0]
Test data generation complete!
```

## **Exercise 8: Compute Relative Accuracy**

In the cell below write the code to perform a two-part evaluation of a `PyTorch model` and its `ONNX-converted version. In Part 1 perform a Single-Sample Equivalence Check. In Part 2 do an accuracy comparison on the full test set.

In [ ]:
# @title Exercise 8: Compute relative accuracy




If the code is correct you should see something _similar_ to the following output

```text
Verifying model equivalence on single sample...
ONNX prediction shape: (1, 5)
PyTorch prediction shape: (1, 5)
Max difference: 8.34465e-07
Mean difference: 3.643334e-07
Results are identical (within floating point precision): True

Comparing model accuracies on the generated test dataset...
Getting predictions from PyTorch model...
Getting predictions from ONNX model...
PyTorch model accuracy: 0.2100
ONNX model accuracy: 0.2100
Difference in accuracy: 0.000000

--- Verification ---
Number of test samples: 100
PyTorch predictions shape: (100, 5)
ONNX predictions shape: (100, 5)

Sample predictions comparison:
Sample 0: True=0, PyTorch=0, ONNX=0
Sample 1: True=1, PyTorch=0, ONNX=0
Sample 2: True=0, PyTorch=0, ONNX=0
Sample 3: True=3, PyTorch=0, ONNX=0
Sample 4: True=0, PyTorch=0, ONNX=0
```

Again, the PyTorch and the ONNX models are virtually the same.

## **Exercise 9: Visualize Confusion Matrices**

In the cell below write the code to generate two side-by-side Confusion Matrices for the PyTorch model on the left and the converted ONNX model on the right.



In [ ]:
# @title Exercise 9: Visualize Similarities with Confusion Plots




If your code is correct you should see something _similar_ to the following output

![__](https://biologicslab.co/BIO1173/images/class_06/class_06_2_image04A.png)

## **Exercise 10: Save ONNX Model to Google Drive**

Run the next cell to save your retrained `ResNet101_model_512` model to your Google Drive.

**IMPORTANT NOTE** You will be using this saved ONNX model in a latter class lesson so make sure **not** to delete it from your Google Drive!

In [ ]:
# @title Exercise 10: Save ONNX Model to Google Drive



If your code is correct, you should see something _similar_ to the following output:
```text
Copying /content/ResNet101_model_512.onnx to Google Drive...
ONNX model copied successfully!
Drive copy present: True
total 2.4G
drwx------ 2 root root 4.0K Apr  5  2024 'Colab Notebooks'
drwx------ 2 root root 4.0K Aug 18  2025  ResNet101_model_512
-rw------- 1 root root 443K Apr 12 18:49  ResNet101_model_512.onnx
drwx------ 2 root root 4.0K Aug 21  2025  ResNet50_model_244
drwx------ 2 root root 4.0K Apr 12 18:30  ResNet50_model_244.onnx

```

## **Class_06_2: Summary**

The primary objective of this class lesson was to convince you that converting a PyTorch model into the ONNX format does **not** have any negative impact on the accuracy of the PyTorch model. As you will see later, there are some compelling reasons why someone migth want to use an ONNX model instead of the "native" PyTorch.

## **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

In [ ]:
# @title  Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/BIO1173/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()